In [3]:
import pandas as pd

df = pd.read_excel('data_2_MCU.xlsx', engine='openpyxl')
missing_value = df.isnull().sum()
cols_to_drop = missing_value[missing_value > 3000].index
df.drop(columns=cols_to_drop, inplace=True)
df.head(10)
df.to_csv(
    'data_2_MCU.csv', index= False
)
df.set_index('TANGGAL', inplace=True)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.to_list()
cat_df = pd.DataFrame({'Column': categorical_cols, 'Unique Values': [df[col].nunique() for col in categorical_cols]}).set_index('Column')
cat_df.sort_values(by='Unique Values', ascending=False, inplace=True)
cat_binary_encode = [col for col in categorical_cols if cat_df.loc[col, 'Unique Values'] <= 2]
cat_df_to_drop = [col for col in categorical_cols if cat_df.loc[col, 'Unique Values'] > 100]
df.drop(columns=cat_df_to_drop, inplace=True)
df.sample(10)


,BADGE,KEHAMILAN,OLAHRAGA,ALERGI,TINGGI,BERAT,TENSI,NADI,PERNAPASAN,SUHU,...,TRIGLISERIDA,HDL_KOLEST,LDL_KOLEST,UREUM,KREATININ,ASAM_URAT_GINJAL,GULA_DARAH_PUASA,GULA_DARAH_2JAMPP,URINE_REDUKSI_PUASA,URINE_REDUKSI_2JAMPP
TANGGAL,,,,,,,,,,,,,,,,,,,,,
2025-01-06,133084880.0,-,+,-,1630.0,680.0,120/80,700.0,200.0,36500.0,...,1350.0,440.0,1550.0,300.0,10.0,5700.0,970.0,820.0,Negatif,Negatif
2024-01-18,133192170.0,-,+,-,1730.0,800.0,150/90,730.0,200.0,360.0,...,1830.0,490.0,1710.0,310.0,1200.0,8500.0,870.0,880.0,Negatif,Negatif
2025-06-13,132492240.0,-,-,-,1480.0,510.0,120/80,770.0,200.0,360.0,...,1240.0,510.0,1140.0,140.0,600.0,4900.0,900.0,1020.0,Negatif,Negatif
2023-10-18,133596230.0,-,+,-,1780.0,780.0,120/80,630.0,200.0,360.0,...,1140.0,550.0,1170.0,160.0,10.0,6900.0,950.0,1140.0,Negatif,Negatif
2024-07-18,132779070.0,-,+,-,1640.0,760.0,120/90,660.0,200.0,360.0,...,790.0,450.0,1280.0,230.0,10.0,4500.0,890.0,1250.0,Negatif,Negatif
2024-02-05,133309290.0,-,-,-,1570.0,540.0,100/70,790.0,200.0,360.0,...,840.0,580.0,830.0,110.0,600.0,3200.0,990.0,940.0,Negatif,Negatif
2023-08-14,133493230.0,-,+,-,1700.0,750.0,120/70,730.0,200.0,360.0,...,1880.0,520.0,1940.0,190.0,1100.0,8300.0,920.0,1050.0,Negatif,Negatif
2024-09-30,132807640.0,-,+,-,1650.0,650.0,130/80,730.0,200.0,360.0,...,2070.0,540.0,1490.0,200.0,800.0,8500.0,1030.0,880.0,Negatif,Negatif
2025-07-24,132902060.0,-,+,-,1750.0,800.0,130/80,790.0,200.0,36500.0,...,1730.0,450.0,2320.0,260.0,900.0,10100.0,1000.0,980.0,Negatif,Negatif


In [4]:
df.drop(columns= [col for col in cat_binary_encode if df[col].nunique()==1], inplace=True)

numeric_cols = df.select_dtypes(include=['number']).columns.to_list()
categorical_cols = df.select_dtypes(include='object').columns.tolist()
agg_df = df.copy()
import category_encoders as ce

cat_cols = agg_df.select_dtypes(include=['object', 'category']).columns.tolist()

for col in cat_cols:
    agg_df[col] = agg_df[col].astype(str).str.strip()
    agg_df[col] = agg_df[col].replace({
        'tidak diperiksa': 'Tidak diperiksa',
        'tidak periksa': 'Tidak diperiksa',
        'tidak di periksa': 'Tidak diperiksa',
        'tidak diperiksa ': 'Tidak diperiksa'
    })

print("BINARY COLUMNS (nunique == 2)")
binary_cols = {}
for col in cat_cols:
    unique_vals = agg_df[col].dropna().unique()
    if len(unique_vals) == 2:
        binary_cols[col] = unique_vals.tolist()
        print(col, unique_vals.tolist())

print("\nMULTI-CARDINALITY COLUMNS (nunique > 2)")
multi_card_cols = {}
for col in cat_cols:
    unique_vals = agg_df[col].dropna().unique()
    if len(unique_vals) > 2:
        multi_card_cols[col] = unique_vals.tolist()
        print(col, unique_vals.tolist())

binary_mapping = {
    'OLAHRAGA': {'+': 1, '-': 0},
    'ALERGI': {'+': 1, '-': 0},
    'CONJUNGTIVA': {'normal': 0, 'anemis': 1},
    'HAEMORROID': {'normal': 0, 'ada': 1},
    'LEHER': {'normal': 0, 'pembesaran KGB': 1},
    'IRAMA_JANTUNG': {'normal': 0, 'Ireguler': 1},
    'TREMOR': {'normal': 0, 'ada': 1},
    'PERUT': {'normal': 0, 'nyeri ketok VA kiri': 1},
    'LIMPA': {'normal': 0, 'teraba': 1},
    'UROBILINOGEN': {'Negatif': 1, 'Tidak periksa': 0}
}

ordinal_mapping = {
    'ERITROSIT_RBC': {'Negatif': 0, '+': 1, '++': 2, '+++': 3},
    'LEKOSIT_WBC': {'Negatif': 0, '+': 1, '++': 2, '+++': 3},
    'SEL_EPITEL': {'Negatif': 0, '+': 1, '++': 2, '+++': 3},
}

for col, mapping in binary_mapping.items():
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].map(mapping)

for col, mapping in ordinal_mapping.items():
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].map(mapping)

remaining_cat_cols = agg_df.select_dtypes(include=['object', 'category']).columns.tolist()
if remaining_cat_cols:
    encoder = ce.BinaryEncoder(cols=remaining_cat_cols, return_df=True)
    agg_df = encoder.fit_transform(agg_df)


BINARY COLUMNS (nunique == 2)
OLAHRAGA ['+', '-']
JVP ['normal', 'meningkat']
HATI ['normal', 'teraba']
LIMPA ['normal', 'teraba']

MULTI-CARDINALITY COLUMNS (nunique > 2)
KEHAMILAN ['-', 'nan', 'G3P2A0 hamil 26 mgg', 'G1P0A0 hamil 15 mgg', 'G3 P1 A1', 'G3 P2 A0', 'G5P2A2 hamil 7 mgg', 'G4 P2 A1', 'G1P0A0 hamil 25 mgg', 'G2P1A0 hamil 31 mgg (+)', 'G2 P1 A0', 'G1 P0 A0', 'G1PoAo ( hamil 24 minggu )', 'G3P2A0 hamil 8 mggu', 'G2 P 1 A0 (Sedang hamil)', 'G4 P3 A0', 'G2P1A0', 'G2P2A0 Hamil 28 mgu', 'G3P2A0 hamil 30 mggu', 'G5P3A1 Hamil 25-26 minggu JTH', 'Riwayat KB IUD (+)', 'G4P2A1 hamil 28-29 mggu', 'G2P1A0 hamil 29 mggu (rutin kontrol dr.mustofa SpOG)', 'Post Partum 4 bulan yll', 'G1P0A0 Hamil 24 minggu', 'G5P4A0 hamil 22 minggu', 'G4P2A1 hamil 37 minggu', 'G5P1A3 Hamil 4 minggu', 'G3P1A1 hamil 14 minggu', 'G1P0A0 Hamil 11 Minggu', 'Operasi SC 2x', 'G4P2A1 Hamil 17 minggu', 'G2P0A1 Hamil 7 bulan']
ALERGI ['-', '+', '+ (steroid tetes mata )']
TENSI ['110/80', '120/70', '110/60', '110/70'

In [5]:
import pandas as pd
import numpy as np
import re
import category_encoders as ce

encoded_df = agg_df.copy()

# ===================================================================
# 1. TENSI → SISTOLIK & DIASTOLIK
# ===================================================================
def parse_tensi(x):
    if pd.isna(x): return np.nan, np.nan
    s = str(x).strip().replace(' ', '')
    if '/' not in s: return np.nan, np.nan
    a, b = s.split('/', 1)
    try:
        return float(a) if a else np.nan, float(b) if b else np.nan
    except:
        return np.nan, np.nan

if 'TENSI' in encoded_df.columns:
    encoded_df[['SISTOLIK', 'DIASTOLIK']] = encoded_df['TENSI'].apply(
        lambda x: pd.Series(parse_tensi(x))
    )
    encoded_df.drop('TENSI', axis=1, inplace=True)

# ===================================================================
# 2. KEHAMILAN → TRIMESTER (0 = tidak hamil, 1/2/3)
# ===================================================================
def parse_trimester(x):
    if pd.isna(x): return 0
    s = str(x).lower()
    if '-' in s or 'none' in s or not s: return 0
    m = re.search(r'(\d+)\s*(minggu|mgg|bulan)', s)
    if not m: return 0
    num = int(m.group(1))
    weeks = num * 4.3 if 'bulan' in m.group(2) else num
    return 1 if weeks <= 13 else 2 if weeks <= 28 else 3

if 'KEHAMILAN' in encoded_df.columns:
    encoded_df['TRIMESTER'] = encoded_df['KEHAMILAN'].apply(parse_trimester)
    encoded_df.drop('KEHAMILAN', axis=1, inplace=True)

# ===================================================================
# 3. URINE MICROSCOPIC (ERITROSIT_RBC, LEKOSIT_WBC, SEL_EPITEL)
# ===================================================================
def parse_microscopic(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().lower()
    if s in ['', 'none', 'tidak periksa', 'tidak_diperiksa', '-', 'tidak diperiksa']:
        return np.nan

    # range atau angka tunggal
    nums = re.findall(r'\d+\.?\d*', s)
    if nums:
        nums = [float(n) for n in nums]
        return np.mean(nums)

    # kata-kata umum
    mapping = {
        'nihil':0, 'tidak ada':0, 'negatif':0,
        'sedikit':1, 'rar':1,
        '+':3, '++':8, '+++':15,
        'banyak':12, 'penuh':20, 'padat':20
    }
    return mapping.get(s, np.nan)

urin_cols = ['ERITROSIT_RBC', 'LEKOSIT_WBC', 'SEL_EPITEL']
for c in urin_cols:
    if c in encoded_df.columns:
        encoded_df[c] = encoded_df[c].apply(parse_microscopic)

# ===================================================================
# 4. SEMUA KOLOM OBJECT → lower + strip + standarisasi "tidak diperiksa"
# ===================================================================
obj_cols = encoded_df.select_dtypes(include=['object', 'category']).columns
for c in obj_cols:
    encoded_df[c] = (encoded_df[c]
                     .astype(str)
                     .str.strip()
                     .str.lower()
                     .replace({'tidak diperiksa':'tidak_diperiksa',
                               'tidak periksa':'tidak_diperiksa',
                               'vtidak diperiksa':'tidak_diperiksa',
                               'none':'tidak_diperiksa',
                               'nan':'tidak_diperiksa',
                               'tidak melakukan':'tidak_diperiksa'}))

# ===================================================================
# 5. BINARY & ORDINAL MAPPING (sesuai nilai unik yang kamu berikan)
# ===================================================================
binary_map = {
    'OLAHRAGA'                   : {'+':1, '-':0, 'tidak_diperiksa':np.nan},
    'ALERGI'                     : {'+':1, '-':0, 'tidak_diperiksa':np.nan},
    'CONJUNGTIVA'                : {'normal':0, 'anemis':1, 'tidak_diperiksa':np.nan},
    'HAEMORROID'                 : {'normal':0, 'ada':1, 'tidak_diperiksa':np.nan},
    'EPIDIDYMIS_TESTIS_PROSTAT'  : {'normal':0, 'tidak_diperiksa':np.nan},
    'LEHER'                      : {'normal':0, 'pembesaran kgb':1, 'tidak_diperiksa':np.nan},
    'IRAMA_JANTUNG'              : {'normal':0, 'ireguler':1, 'tidak_diperiksa':np.nan},
    'TREMOR'                     : {'normal':0, 'ada':1, 'tidak_diperiksa':np.nan},
    'PERUT'                      : {'normal':0, 'nyeri ketok va kiri':1, 'tidak_diperiksa':np.nan},
    'LIMPA'                      : {'normal':0, 'teraba':1, 'tidak_diperiksa':np.nan},
}

ordinal_map = {
    'UROBILINOGEN'    : {'negatif':0, 'tidak_diperiksa':np.nan},
    'BILIRUBIN'       : {'negatif':0, 'tidak_diperiksa':np.nan},
    'ASAM_URAT_URIN'  : {'negatif':0, 'tidak_diperiksa':np.nan},
    'TRIPLE_PHOSP'    : {'negatif':0, 'tidak_diperiksa':np.nan},
    'HYALINE'         : {'negatif':0, 'tidak_diperiksa':np.nan},
    'PROTEIN_ALBUMIN' : {'negatif':0, 'positif 1':1, 'positif':2, 'tidak_diperiksa':np.nan},
    'REDUKSI'         : {'negatif':0, 'positif 1':1, 'positif 2':2, 'positif':2, 'tidak_diperiksa':np.nan},
    'AMORF'           : {'negatif':0, 'positif':1, 'tidak_diperiksa':np.nan},
    'CA_OX'           : {'negatif':0, 'positif':1, 'tidak_diperiksa':np.nan},
    'GRANULER'        : {'negatif':0, 'positif':1, 'tidak_diperiksa':np.nan},
    'BAKTERI'         : {'negatif':0, 'positif':1, 'positif 2':2, 'tidak_diperiksa':np.nan},
    'URINE_REDUKSI_PUASA': {'negatif':0, 'positif1':1, 'positif 1':1, 'positif 2':2, 'positif':2, 'tab':np.nan, 'tidak_diperiksa':np.nan},
    'URINE_REDUKSI_2JAMPP': {'negatif':0, 'positif 1':1, 'positif 2':2, 'positif 3':3, 'positif':2, 'tab':np.nan, 'tidak_diperiksa':np.nan},
}

# terapkan mapping (aman untuk kolom yang sudah numerik)
for col, mp in {**binary_map, **ordinal_map}.items():
    if col in encoded_df.columns:
        # jika sudah 0/1, biarkan
        if encoded_df[col].dtype in ['int64','float64'] and encoded_df[col].dropna().isin([0,1]).all():
            continue
        encoded_df[col] = encoded_df[col].map(mp)

# ===================================================================
# 6. KOLOM LAIN YANG MASIH KATEGORIKAL → Binary Encoding
# ===================================================================
# hapus kolom yang seluruhnya NaN dulu
cat_cols = encoded_df.select_dtypes(include=['object','category']).columns
encoded_df.drop(columns=[c for c in cat_cols if encoded_df[c].isna().all()], inplace=True)

# binary encoding untuk sisanya
remaining_cat = encoded_df.select_dtypes(include=['object','category']).columns.tolist()
if remaining_cat:
    encoder = ce.BinaryEncoder(cols=remaining_cat, return_df=True)
    encoded_df = encoder.fit_transform(encoded_df)

# ===================================================================
# DONE
# ===================================================================
print("Encoding selesai tanpa warning!")
print(f"Shape akhir: {encoded_df.shape}")
print(f"Missing di urin cols:")
print(encoded_df[['ERITROSIT_RBC','LEKOSIT_WBC','SEL_EPITEL']].isna().sum())

Encoding selesai tanpa warning!
Shape akhir: (10637, 210)
Missing di urin cols:
ERITROSIT_RBC    10637
LEKOSIT_WBC      10637
SEL_EPITEL       10637
dtype: int64


In [6]:
encoded_df = encoded_df.drop(columns=['ERITROSIT_RBC', 'LEKOSIT_WBC', 'SEL_EPITEL'])
encoded_df = encoded_df.drop(columns=['KEHAMILAN_0', 'KEHAMILAN_1', 'KEHAMILAN_2', 'KEHAMILAN_3', 'KEHAMILAN_4'])
encoded_df.sample(10)


,BADGE,KEHAMILAN_5,OLAHRAGA,ALERGI,TINGGI,BERAT,TENSI_0,TENSI_1,TENSI_2,TENSI_3,...,GULA_DARAH_2JAMPP,URINE_REDUKSI_PUASA_0,URINE_REDUKSI_PUASA_1,URINE_REDUKSI_PUASA_2,URINE_REDUKSI_PUASA_3,URINE_REDUKSI_2JAMPP_0,URINE_REDUKSI_2JAMPP_1,URINE_REDUKSI_2JAMPP_2,URINE_REDUKSI_2JAMPP_3,URINE_REDUKSI_2JAMPP_4
TANGGAL,,,,,,,,,,,,,,,,,,,,,
2020-10-01,132777750.0,1,1,0.0,1760.0,720.0,0,0,0,0,...,800.0,0,0,0,1,0,0,0,0,1
2020-06-19,133308620.0,1,0,0.0,1620.0,620.0,0,0,0,0,...,880.0,0,0,0,1,0,0,0,0,1
2022-03-17,132692610.0,1,0,0.0,1690.0,740.0,0,0,0,0,...,860.0,0,0,0,1,0,0,0,0,1
2021-09-06,132799960.0,1,1,0.0,1760.0,700.0,0,0,0,1,...,790.0,0,0,0,1,0,0,0,0,1
2020-03-09,132689550.0,1,0,0.0,1640.0,760.0,0,0,0,0,...,1070.0,0,0,0,1,0,0,0,0,1
2020-06-25,133305320.0,1,0,0.0,1610.0,640.0,0,0,0,1,...,880.0,0,0,0,1,0,0,0,0,1
2024-06-07,132691550.0,1,1,0.0,1540.0,740.0,0,1,0,0,...,1070.0,0,0,0,1,0,0,0,0,1
2022-01-27,134223720.0,1,0,0.0,1640.0,790.0,0,0,0,1,...,1100.0,0,0,0,1,0,0,0,0,1
2024-01-23,133307330.0,1,0,0.0,1740.0,840.0,0,0,0,1,...,960.0,0,0,0,1,0,0,0,0,1


In [7]:
sample_data = encoded_df.loc[:0]

# 1. Mengubah row menjadi list nilai
sample_list = sample_data.values.tolist()[0]  # ambil [0] karena cuma 1 row
print(sample_list)

TypeError: cannot do slice indexing on DatetimeIndex with these indexers [0] of type int

In [9]:
# CELL BARU — AGGREGATE BINARY ENCODED COLUMNS SUPAYA LEBIH RINGKAS & BAGUS BUAT CLUSTERING
final_df = encoded_df.copy()

# ===================================================================
# 1. BUANG KOLOM YANG HAMPIR SEMUA 0 (rare event) → tidak informatif
# ===================================================================
rare_threshold = 0.01  # <1% yang 1 → buang
rare_cols = final_df.columns[final_df.mean() < rare_threshold]
print(f"Buang {len(rare_cols)} kolom rare (<1% positif): {list(rare_cols)[:10]}...")
final_df = final_df.drop(columns=rare_cols)

# ===================================================================
# 2. GABUNG BINARY ENCODED KOLOM YANG SEJENIS (MANUAL GROUPING LOGIS)
# ===================================================================
# Kelompokkan berdasarkan organ / pemeriksaan
groupings = {
    # TELINGA & TURUNANNYA
    'ANY_EAR_ABNORMAL'       : ['TELINGA_1', 'MEMBRAN_TYMPANI_1', 'MEMBRAN_TYMPANI_2', 'SERUMEN_PLUG_1', 'SERUMEN_PLUG_2', 'SERUMEN_PLUG_3', 'SERUMEN_PLUG_4'],
    # HIDUNG & TURUNANNYA
    'ANY_NOSE_ABNORMAL'      : ['HIDUNG_1', 'SEPTUM_DEVIASI_1', 'SEPTUM_DEVIASI_2', 'CONCHA_1', 'CONCHA_2', 'POLYP_1', 'POLYP_2'],
    # MULUT & GIGI
    'ANY_ORAL_ABNORMAL'      : ['MULUT_1', 'MULUT_2', 'GUSI_1'],
    # TENGGOROKAN
    'ANY_THROAT_ABNORMAL'    : ['KERONGKONGAN_1', 'TONSIL_1', 'TONSIL_2', 'TONSIL_3', 'TONSIL_4', 'FARING_1', 'FARING_2'],

    # URINE SEDIMEN (non-negatif)
    'ANY_URINE_SEDIMENT_POS' : ['PROTEIN_ALBUMIN_1', 'PROTEIN_ALBUMIN_2', 'REDUKSI_1', 'REDUKSI_2', 
                                'AMORF_1', 'CA_OX_1', 'CA_OX_2', 'GRANULER_1', 'HYALINE_1', 
                                'BAKTERI_1', 'BAKTERI_2'],
    # GULA URINE (puasa / 2jam pp)
    'ANY_GLUCOSE_URINE_POS'  : ['URINE_REDUKSI_PUASA_1', 'URINE_REDUKSI_PUASA_2', 
                                'URINE_REDUKSI_2JAMPP_1', 'URINE_REDUKSI_2JAMPP_2', 'URINE_REDUKSI_2JAMPP_3'],
}

for new_col, old_cols in groupings.items():
    cols_exist = [c for c in old_cols if c in final_df.columns]
    if cols_exist:
        final_df[new_col] = final_df[cols_exist].max(axis=1)  # 1 jika ada salah satu abnormal
        final_df = final_df.drop(columns=cols_exist)

# ===================================================================
# 3. KEEP YANG PENTING & SUDAH BAGUS (binary langsung)
# ===================================================================
keep_direct = ['OLAHR conlusion', 'ALERGI', 'CONJUNGTIVA', 'HAEMORROID', 'LEHER', 'IRAMA_JANTUNG', 
               'TREMOR', 'PERUT', 'LIMPA', 'HERNIA_0', 'EPIDIDYMIS_TESTIS_PROSTAT_0', 
               'UROBILINOGEN', 'BILIRUBIN_0', 'ASAM_URAT_URIN_0', 'TRIPLE_PHOSP_0']

keep_direct = [c for c in keep_direct if c in final_df.columns]

# ===================================================================
# 4. FINAL SELECTION: numeric + lab + yang di-keep + yang baru digabung
# ===================================================================
numeric_lab_cols = ['TINGGI', 'BERAT', 'NADI', 'PERNAPASAN', 'SUHU', 'HB', 'LEUKOSIT', 
                    'LED', 'EOSINOPIL', 'BASOPIL', 'SEGMENT', 'LYMPOSIT', 'MONOSIT', 'TROMBOSIT',
                    'BILIRUBIN_TOTAL', 'BILIRUBIN_DIRECT', 'BILIRUBIN_INDIRECT', 'ALKALINE_PHOSPAT',
                    'SGPT', 'SGOT', 'GAMMA_GT', 'KOLEST_TOTAL', 'TRIGLISERIDA', 'HDL_KOLEST', 
                    'LDL_KOLEST', 'UREUM', 'KREATININ', 'ASAM_URAT_GINJAL', 
                    'GULA_WAKTU', 'GULA_DARAH_2JAMPP', 
                    'SISTOLIK', 'DIASTOLIK', 'ERITROSIT_RBC', 'LEKOSIT_WBC', 'SEL_EPITEL']

numeric_lab_cols = [c for c in numeric_lab_cols if c in final_df.columns]

# Gabungkan semua
selected_cols = numeric_lab_cols + keep_direct + list(groupings.keys())
final_clustering_df = final_df[selected_cols].copy()

# Optional: tambah BMI
if 'TINGGI' in final_clustering_df.columns and 'BERAT' in final_clustering_df.columns:
    final_clustering_df['BMI'] = final_clustering_df['BERAT'] / ((final_clustering_df['TINGGI']/100)**2)

# Optional: tambah MAP (Mean Arterial Pressure)
if 'SISTOLIK' in final_clustering_df.columns and 'DIASTOLIK' in final_clustering_df.columns:
    final_clustering_df['MAP'] = final_clustering_df['DIASTOLIK'] + (final_clustering_df['SISTOLIK'] - final_clustering_df['DIASTOLIK']) / 3

print("SELESAI! Kolom siap clustering:")
print(f"→ Dari {encoded_df.shape[1]} kolom → {final_clustering_df.shape[1]} kolom")
print(f"→ Kolom akhir: {list(final_clustering_df.columns)}")

Buang 87 kolom rare (<1% positif): ['TENSI_0', 'KULIT_RAMBUT_0', 'KULIT_RAMBUT_1', 'PENYAKIT_MATA_0', 'PENYAKIT_MATA_1', 'SCLERA_0', 'SCLERA_1', 'TELINGA_0', 'TELINGA_1', 'TELINGA_2']...
SELESAI! Kolom siap clustering:
→ Dari 202 kolom → 41 kolom
→ Kolom akhir: ['TINGGI', 'BERAT', 'NADI', 'PERNAPASAN', 'SUHU', 'HB', 'LEUKOSIT', 'LED', 'EOSINOPIL', 'BASOPIL', 'SEGMENT', 'LYMPOSIT', 'MONOSIT', 'TROMBOSIT', 'BILIRUBIN_TOTAL', 'BILIRUBIN_DIRECT', 'BILIRUBIN_INDIRECT', 'ALKALINE_PHOSPAT', 'SGPT', 'SGOT', 'GAMMA_GT', 'KOLEST_TOTAL', 'TRIGLISERIDA', 'HDL_KOLEST', 'LDL_KOLEST', 'UREUM', 'KREATININ', 'ASAM_URAT_GINJAL', 'GULA_DARAH_2JAMPP', 'ALERGI', 'CONJUNGTIVA', 'HAEMORROID', 'EPIDIDYMIS_TESTIS_PROSTAT_0', 'UROBILINOGEN', 'ANY_EAR_ABNORMAL', 'ANY_NOSE_ABNORMAL', 'ANY_ORAL_ABNORMAL', 'ANY_THROAT_ABNORMAL', 'ANY_URINE_SEDIMENT_POS', 'ANY_GLUCOSE_URINE_POS', 'BMI']


In [10]:
one_type_only = [loc for loc in final_df.columns if final_df[loc].nunique()==1]
final_df = final_df.drop(columns=one_type_only)
final_df.sample(10)

,BADGE,KEHAMILAN_5,OLAHRAGA,ALERGI,TINGGI,BERAT,TENSI_1,TENSI_2,TENSI_3,TENSI_4,...,GULA_DARAH_PUASA,GULA_DARAH_2JAMPP,URINE_REDUKSI_PUASA_3,URINE_REDUKSI_2JAMPP_4,ANY_EAR_ABNORMAL,ANY_NOSE_ABNORMAL,ANY_ORAL_ABNORMAL,ANY_THROAT_ABNORMAL,ANY_URINE_SEDIMENT_POS,ANY_GLUCOSE_URINE_POS
TANGGAL,,,,,,,,,,,,,,,,,,,,,
2023-06-05,133308700.0,1,1,0.0,1620.0,660.0,0,0,0,0,...,840.0,850.0,1,1,0,0,1,1,1,0
2023-05-16,132498100.0,1,1,0.0,1670.0,600.0,0,0,0,1,...,880.0,850.0,1,1,0,0,1,1,1,0
2020-11-16,132799540.0,1,1,0.0,1660.0,730.0,0,0,0,1,...,830.0,1180.0,1,1,1,1,1,1,1,0
2024-09-09,133002310.0,1,1,0.0,1730.0,770.0,0,0,1,0,...,900.0,990.0,1,1,0,0,0,1,1,0
2024-09-20,133586620.0,1,1,0.0,1720.0,600.0,0,0,0,1,...,890.0,720.0,1,1,0,0,0,1,1,0
2024-12-04,133588210.0,1,1,0.0,1600.0,720.0,0,0,1,0,...,880.0,960.0,1,1,0,0,1,1,1,0
2020-06-02,166908310.0,1,0,0.0,1730.0,610.0,0,1,0,0,...,890.0,850.0,1,1,1,1,1,1,1,0
2022-10-15,134221820.0,1,1,0.0,1590.0,650.0,0,0,0,1,...,870.0,990.0,1,1,0,0,1,1,1,0
2022-11-14,134219920.0,1,1,0.0,1570.0,570.0,0,1,0,1,...,860.0,1050.0,1,1,0,0,1,1,1,0


In [11]:
null_columns = [col for col in final_df.columns if final_df[col].isnull().sum() > 0]

for col in null_columns:
    if final_df[col].dtype in ['int64', 'float64']:  # numerik
        median_value = final_df[col].median()
        final_df[col].fillna(median_value, inplace=True)
    else:  # categorical / binary
        mode_value = final_df[col].mode()[0]
        final_df[col].fillna(mode_value, inplace=True)
final_df

C:\Users\Loq Gaming\AppData\Local\Temp\ipykernel_38424\1616573314.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_df[col].fillna(median_value, inplace=True)


,BADGE,KEHAMILAN_5,OLAHRAGA,ALERGI,TINGGI,BERAT,TENSI_1,TENSI_2,TENSI_3,TENSI_4,...,GULA_DARAH_PUASA,GULA_DARAH_2JAMPP,URINE_REDUKSI_PUASA_3,URINE_REDUKSI_2JAMPP_4,ANY_EAR_ABNORMAL,ANY_NOSE_ABNORMAL,ANY_ORAL_ABNORMAL,ANY_THROAT_ABNORMAL,ANY_URINE_SEDIMENT_POS,ANY_GLUCOSE_URINE_POS
TANGGAL,,,,,,,,,,,,,,,,,,,,,
2020-01-02,133191500.0,1,1,0.0,1660.0,800.0,0,0,0,0,...,860.0,870.0,1,1,0,0,0,0,1,0
2020-01-03,133313090.0,1,0,0.0,1600.0,570.0,0,0,0,0,...,930.0,1050.0,1,1,0,0,0,1,1,0
2020-01-03,133313090.0,1,0,0.0,1550.0,520.0,0,0,0,1,...,880.0,930.0,1,1,0,0,0,1,1,0
2020-01-03,166996720.0,1,1,0.0,1590.0,570.0,0,0,0,1,...,1250.0,1970.0,1,1,0,0,0,1,1,0
2020-01-03,166281850.0,1,1,0.0,1710.0,620.0,0,0,1,0,...,1070.0,1020.0,1,1,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-23,132901160.0,1,1,0.0,1650.0,730.0,0,0,0,1,...,980.0,970.0,1,1,0,0,0,1,1,0
2025-07-23,132901160.0,1,1,0.0,1570.0,580.0,0,0,0,1,...,920.0,870.0,1,1,0,0,0,1,1,0
2025-07-24,133401860.0,1,1,0.0,1720.0,820.0,0,0,0,1,...,970.0,950.0,1,1,0,0,0,1,1,0


In [12]:
final_df.isnull().sum()

BADGE                     0
KEHAMILAN_5               0
OLAHRAGA                  0
ALERGI                    0
TINGGI                    0
                         ..
ANY_NOSE_ABNORMAL         0
ANY_ORAL_ABNORMAL         0
ANY_THROAT_ABNORMAL       0
ANY_URINE_SEDIMENT_POS    0
ANY_GLUCOSE_URINE_POS     0
Length: 100, dtype: int64

In [13]:
final_df.to_csv('Cleaned_pasien_large.csv', index=False)